# LangGraph Time Travel: Practice Exercise

Practice using LangGraph's time travel feature to go back in an agent's execution history, modify state, and resume from a checkpoint. You will work with a recipe finder workflow and add dietary preferences mid-execution.

**What you'll implement:**
- Creating a checkpointer and compiling the workflow
- Finding the target checkpoint before recipe search query generation
- Updating state at that checkpoint to add a dietary preference
- Resuming execution from the modified checkpoint

**Estimated time:** 10-15 minutes

## Setup

Run the following cell to import all required libraries and initialize the workflow components.

In [ ]:
# Setup - run this cell first

import os
from typing import TypedDict, List, Dict, Optional

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Missing OPENAI_API_KEY")

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

print("Setup complete!")

## Scenario

You have a recipe finder workflow that:
1. Takes a dish request (e.g., "pasta dinner ideas")
2. Generates search queries for recipes
3. Simulates finding recipes based on those queries
4. Produces recipe recommendations

The workflow has an optional `dietary_preference` field (e.g., "vegan", "keto", "gluten-free") that influences query generation.

**Your task:**
1. Run the workflow once without a dietary preference
2. Retrieve the execution history
3. Find the checkpoint right before query generation (where `next` equals `('generate_queries',)`)
4. Update state at that checkpoint to add a dietary preference
5. Resume execution and observe how the queries change

**Inputs:**
- `dish_request`: A string like "pasta dinner ideas"
- `dietary_preference`: Optional string like "vegan" or "keto"

**Expected behavior:**
- Without dietary preference: General recipe queries
- With dietary preference: Queries focused on that diet (e.g., "vegan pasta recipes")

## State Schema and Workflow Nodes

The state and workflow nodes are provided for you. Review them to understand the structure.

In [ ]:
# State schema for recipe finder
class RecipeState(TypedDict):
    """State for recipe finder workflow."""
    dish_request: str                    # What the user wants to cook
    dietary_preference: Optional[str]    # Optional: vegan, keto, gluten-free, etc.
    search_queries: List[str]            # Generated search queries
    found_recipes: Dict[str, str]        # Query -> recipe summary
    recommendations: str                 # Final recommendations


def parse_request_node(state: RecipeState) -> RecipeState:
    """Validate request and initialize state."""
    request = state.get("dish_request", "").strip()
    if not request:
        raise ValueError("Dish request cannot be empty")
    
    dietary = state.get("dietary_preference")
    if dietary:
        print(f"[parse_request] Dietary preference: {dietary}")
    else:
        print("[parse_request] No dietary preference")
    
    return {
        "dish_request": request,
        "dietary_preference": dietary,
        "search_queries": [],
        "found_recipes": {},
        "recommendations": ""
    }


def generate_queries_node(state: RecipeState) -> RecipeState:
    """Generate search queries, adapting to dietary preference if present."""
    request = state["dish_request"]
    dietary = state.get("dietary_preference")
    
    if dietary:
        print(f"[generate_queries] Using dietary preference: {dietary}")
        prompt = f"""Generate 3 recipe search queries for: {request}
IMPORTANT: All queries must be for {dietary} recipes only.
Return one query per line, no numbering."""
    else:
        print("[generate_queries] No dietary preference - general queries")
        prompt = f"""Generate 3 diverse recipe search queries for: {request}
Return one query per line, no numbering."""
    
    response = llm.invoke(prompt)
    queries = [q.strip() for q in response.content.strip().split('\n') if q.strip()][:3]
    
    print(f"[generate_queries] Generated queries:")
    for q in queries:
        print(f"  - {q}")
    
    return {
        "dish_request": state["dish_request"],
        "dietary_preference": state.get("dietary_preference"),
        "search_queries": queries,
        "found_recipes": {},
        "recommendations": ""
    }


def find_recipes_node(state: RecipeState) -> RecipeState:
    """Simulate finding recipes (mock search results)."""
    queries = state["search_queries"]
    dietary = state.get("dietary_preference", "any diet")
    
    print(f"[find_recipes] Searching for {len(queries)} queries...")
    
    # Simulated recipe results
    found_recipes = {}
    for i, query in enumerate(queries, 1):
        found_recipes[query] = f"Found 5 {dietary} recipes matching '{query}' with ratings 4-5 stars."
    
    return {
        "dish_request": state["dish_request"],
        "dietary_preference": state.get("dietary_preference"),
        "search_queries": state["search_queries"],
        "found_recipes": found_recipes,
        "recommendations": ""
    }


def recommend_node(state: RecipeState) -> RecipeState:
    """Generate final recommendations."""
    dietary = state.get("dietary_preference")
    dietary_note = f" ({dietary})" if dietary else ""
    
    recommendations = f"Based on your request for '{state['dish_request']}'{dietary_note}, "
    recommendations += f"we searched using {len(state['search_queries'])} queries and found great recipes!"
    
    print(f"[recommend] Generated recommendations")
    
    return {
        "dish_request": state["dish_request"],
        "dietary_preference": state.get("dietary_preference"),
        "search_queries": state["search_queries"],
        "found_recipes": state["found_recipes"],
        "recommendations": recommendations
    }


print("State schema and nodes defined!")

## Build the Workflow Graph

The workflow structure is built for you below. Review the node and edge configuration.

In [ ]:
# Build the workflow graph
workflow = StateGraph(RecipeState)

# Add nodes
workflow.add_node("parse_request", parse_request_node)
workflow.add_node("generate_queries", generate_queries_node)
workflow.add_node("find_recipes", find_recipes_node)
workflow.add_node("recommend", recommend_node)

# Add edges
workflow.add_edge(START, "parse_request")
workflow.add_edge("parse_request", "generate_queries")
workflow.add_edge("generate_queries", "find_recipes")
workflow.add_edge("find_recipes", "recommend")
workflow.add_edge("recommend", END)

print("Workflow graph built!")
print("Flow: parse_request -> generate_queries -> find_recipes -> recommend")

## YOUR TASK: Create Checkpointer and Compile Workflow

To enable time travel, you need to:
1. Create a `MemorySaver` checkpointer instance
2. Compile the workflow with the checkpointer

**Hint:** Use `workflow.compile(checkpointer=your_checkpointer)`

In [ ]:
# TODO: Create a MemorySaver checkpointer
checkpointer = None  # Replace with your implementation

# TODO: Compile the workflow with the checkpointer
recipe_app = None  # Replace with your implementation

print("Workflow compiled with checkpointer!")

## Step 1: Initial Run (No Dietary Preference)

Run the workflow without a dietary preference to establish the baseline execution.

In [ ]:
# Initial state without dietary preference
initial_state = {
    "dish_request": "pasta dinner ideas",
    "dietary_preference": None,
    "search_queries": [],
    "found_recipes": {},
    "recommendations": ""
}

# Config with thread_id for tracking execution history
config = {"configurable": {"thread_id": "recipe_001"}}

# Run the workflow
print("=" * 60)
print("INITIAL RUN: No dietary preference")
print("=" * 60)

initial_result = recipe_app.invoke(initial_state, config)

print("\n" + "=" * 60)
print("INITIAL RESULTS")
print("=" * 60)
print(f"\nDietary preference: {initial_result.get('dietary_preference')}")
print(f"\nSearch queries:")
for q in initial_result["search_queries"]:
    print(f"  - {q}")
print(f"\nRecommendations: {initial_result['recommendations']}")

## Step 2: Retrieve Execution History

Get the list of all checkpoints created during execution.

In [ ]:
# Retrieve execution history
history = list(recipe_app.get_state_history(config))

print(f"Found {len(history)} checkpoints:\n")
for i, checkpoint in enumerate(history):
    print(f"Checkpoint {i}: next={checkpoint.next}")

## TODO: Find the Target Checkpoint

Find the checkpoint where the next node is `generate_queries`. This is right after `parse_request` completed but before query generation started.

In [ ]:
def find_target_checkpoint(history: list) -> object:
    """
    Find the checkpoint right before query generation.
    
    Args:
        history: List of checkpoints from get_state_history()
    
    Returns:
        The checkpoint where next == ('generate_queries',)
    
    Raises:
        ValueError: If target checkpoint not found
    """
    # TODO: Iterate through history and find the checkpoint where
    # checkpoint.next equals ('generate_queries',)
    # Return that checkpoint
    # Raise ValueError if not found
    pass

In [ ]:
# Find and display the target checkpoint
target_checkpoint = find_target_checkpoint(history)

print("Target checkpoint found!")
print(f"  Next node: {target_checkpoint.next}")
print(f"  Current dietary_preference: {target_checkpoint.values.get('dietary_preference')}")
print(f"  Checkpoint ID: {target_checkpoint.config['configurable']['checkpoint_id']}")

## TODO: Update State at Checkpoint (Time Travel)

Add a dietary preference to the state at the target checkpoint. Use `as_node="parse_request"` to indicate that this update came from the parse_request node, so execution will continue with generate_queries.

In [ ]:
def update_with_dietary_preference(app, checkpoint, preference: str) -> dict:
    """
    Update state at checkpoint to add a dietary preference.
    
    Args:
        app: The compiled workflow app
        checkpoint: The target checkpoint to update
        preference: The dietary preference to add (e.g., "vegan")
    
    Returns:
        New config pointing to the updated checkpoint
    
    Hint: Use app.update_state() with:
        - checkpoint.config as the first argument
        - values dict containing the dietary_preference
        - as_node="parse_request" to indicate where to resume from
    """
    # TODO: Call app.update_state() to add the dietary preference
    # and return the new config
    pass

In [ ]:
# Update state with dietary preference
dietary_choice = "vegan"

print(f"Adding dietary preference: '{dietary_choice}'")
new_config = update_with_dietary_preference(recipe_app, target_checkpoint, dietary_choice)

print(f"\nNew checkpoint created!")
print(f"  New checkpoint ID: {new_config['configurable']['checkpoint_id']}")

## TODO: Resume Execution from Modified Checkpoint

Resume the workflow from the modified checkpoint. Pass `None` as the state since we are resuming from a checkpoint.

In [ ]:
def resume_from_checkpoint(app, config: dict) -> dict:
    """
    Resume workflow execution from a checkpoint.
    
    Args:
        app: The compiled workflow app
        config: Config pointing to the checkpoint to resume from
    
    Returns:
        The final result after execution completes
    
    Hint: Use app.invoke() with None as the first argument
    (since we're resuming, not starting fresh)
    """
    # TODO: Resume execution using app.invoke()
    pass

In [ ]:
# Resume execution
print("=" * 60)
print(f"RESUMING WITH DIETARY PREFERENCE: {dietary_choice}")
print("=" * 60)

alternative_result = resume_from_checkpoint(recipe_app, new_config)

print("\n" + "=" * 60)
print("ALTERNATIVE TIMELINE RESULTS")
print("=" * 60)
print(f"\nDietary preference: {alternative_result.get('dietary_preference')}")
print(f"\nSearch queries:")
for q in alternative_result["search_queries"]:
    print(f"  - {q}")
print(f"\nRecommendations: {alternative_result['recommendations']}")

## Compare Results

Run this cell to compare the queries from both timelines.

In [ ]:
# Compare the two timelines
print("=" * 60)
print("TIMELINE COMPARISON")
print("=" * 60)

print("\nORIGINAL (no dietary preference):")
for q in initial_result["search_queries"]:
    print(f"  - {q}")

print(f"\nALTERNATIVE (with '{dietary_choice}'):")
for q in alternative_result["search_queries"]:
    print(f"  - {q}")

print("\n" + "=" * 60)
print("Notice how the queries changed to focus on the dietary preference!")